# 윈도우 메모리 — RunnableWithMessageHistory 기반 Modern 패턴

## 이번 노트북 학습 목표

- `last_k_messages()` 함수로 전체 히스토리에서 최근 k개 메시지만 추출하는 윈도우(window) 패턴을 익혀요.
- 전체 버퍼 체인과 윈도우 체인의 동작 차이를 고객 상담 시나리오로 비교해요.
- 토큰 비용과 맥락 유지 사이의 균형을 어떻게 설계할지 이해해요.

## 사전 지식

- `RunnableWithMessageHistory` 기본 사용법 (10번 노트북)
- Modern 메모리 패턴 개요 (11번 노트북)

## ConversationBufferWindowMemory(구식) vs 윈도우 패턴(Modern) 비교

| 항목 | 구식 (02번, `ConversationBufferWindowMemory`) | Modern (12번, `last_k_messages`) |
|------|----------------------------------------------|----------------------------------|
| 메모리 클래스 | `ConversationBufferWindowMemory(k=3)` | 없음 (함수로 대체) |
| 저장 방식 | 내부 버퍼에 최근 k턴만 보관 | 전체 히스토리 저장 후 프롬프트 시점에 트리밍 |
| 세션 분리 | 별도 구현 필요 | `session_id`로 자유롭게 분리 |
| 백엔드 교체 | 인메모리 전용 | SQLite 등으로 교체 가능 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

> 🎯 **강의 포인트**: Modern 윈도우 패턴의 핵심 차이는 "전체 히스토리는 저장하되, 프롬프트에 넣을 때만 트리밍"한다는 점입니다. Legacy는 메모리 객체 자체에서 오래된 데이터를 버렸지만, Modern에서는 데이터를 보존하고 프롬프트 주입 시점에 조절합니다.

> 💡 **실무 팁**: 한 턴(사용자 + AI)은 메시지 2개입니다. `k=6`이면 최근 3턴에 해당합니다. 서비스 특성에 따라 일반 상담은 k=4~6, 기술 지원처럼 복잡한 대화는 k=8~10을 권장합니다.

> 이 노트북은 `RunnableWithMessageHistory`(실행 가능한 메시지 히스토리 래퍼)와 `last_k_messages()`(k개 메시지 추출 함수)를 조합하는 **Modern 윈도우 패턴**을 다뤄요.

## Before / After 비교: 윈도우 메모리

### Before (Legacy 방식)

```python
# 구식: ConversationBufferWindowMemory 클래스 사용
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=3)  # 최근 3턴만 자동 유지
# → 메모리 객체가 알아서 오래된 대화를 삭제합니다
# → 한번 삭제되면 복구 불가!
```

### After (Modern 방식)

```python
# Modern: 직접 함수로 슬라이싱
def last_k_messages(messages, k=6):
    return messages[-k:]  # 전체 히스토리에서 최근 k개만 추출

# → 원본 히스토리는 그대로 보존
# → 프롬프트에 넣을 때만 잘라서 사용
```

### 비유로 이해하기

| | Legacy | Modern |
|---|---|---|
| 비유 | **자동 슬라이딩 도어** - 문이 알아서 열리고 닫힘. 한번 닫히면 안쪽을 볼 수 없음 | **수동으로 열 수 있는 도어** - 원하는 만큼만 열어서 필요한 것만 꺼냄 |
| 데이터 보존 | 오래된 대화 영구 삭제 | 전체 대화 보존, 보여줄 범위만 조절 |
| 유연성 | k 값 고정 | 상황에 따라 k 값 동적 변경 가능 |

### 언제 윈도우 메모리를 사용하나요?

- **고객 상담 챗봇**: 최근 맥락만 중요하고, 과거 대화는 참고할 필요 없을 때 (예: "배송 어디까지 왔나요?")
- **코딩 어시스턴트**: 최근 몇 번의 코드 수정 내역만 기억하면 될 때
- **일상 대화 봇**: 가벼운 잡담에서 전체 대화를 기억할 필요가 없을 때
- **토큰 비용 절감**: 대화가 길어져도 프롬프트 크기를 일정하게 유지하고 싶을 때

In [1]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()

# LLM 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 기본 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 친절한 고객 상담 AI입니다. 이전 대화 맥락을 활용해 자연스럽게 답변하세요.",
        ),
        MessagesPlaceholder("chat_history"),  # 메모리에서 불러올 대화 기록
        ("human", "{question}"),
    ]
)

# 메모리 기능이 없는 기본 체인
base_chain = prompt | model | StrOutputParser()


# 세션별 대화 기록을 저장할 인메모리 스토어 (실서비스에서는 DB/캐시 사용 권장)
store: dict[str, ChatMessageHistory] = {}


def get_session_history(session_id: str) -> ChatMessageHistory:
    """세션 ID를 기반으로 대화 기록을 가져오는 함수.

    - 없으면 새 `ChatMessageHistory`를 생성해서 store에 등록
    - 있으면 기존 히스토리를 그대로 사용
    """
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 전체 버퍼 메모리를 사용하는 체인 (윈도우 트리밍 없음)
full_buffer_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",      # 사용자의 입력이 들어가는 키
    history_messages_key="chat_history",  # 프롬프트의 MessagesPlaceholder 키
)


## 1. 전체 버퍼 메모리 복습

`RunnableWithMessageHistory`(실행 가능한 메시지 히스토리 래퍼)를 사용해 **전체 대화 이력을 그대로 사용하는 버퍼 메모리**를 먼저 복습해요. 이것이 윈도우 패턴의 출발점이에요.

`full_buffer_chain`은 한 세션의 모든 대화가 순서대로 쌓여요. 대화가 길어질수록 프롬프트도 계속 늘어나요.

### 전체 버퍼 실행 흐름

```mermaid
flowchart LR
    QUESTION[사용자 질문] --> FULL_CHAIN[full_buffer_chain.invoke<br/>+ config session_id]
    FULL_CHAIN --> LOAD_ALL[전체 ChatMessageHistory 로드<br/>누적된 모든 메시지]
    LOAD_ALL --> PROMPT[ChatPromptTemplate<br/>chat_history 전체 주입]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> PARSER[StrOutputParser]
    PARSER --> SAVE[자동 저장<br/>RunnableWithMessageHistory]
    SAVE --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class FULL_CHAIN,PROMPT,LLM,PARSER process
    class LOAD_ALL,SAVE storage
    class ANSWER output
```


In [2]:
# ---------------------------------------------------
# 전체 버퍼 메모리 동작 확인: 모든 대화 이력 사용
# ---------------------------------------------------

# 1단계: config 설정 (동일한 session_id 사용 → 이전 내용을 기억)
# - configurable 딕셔너리 안에 session_id를 넣어야 함
config_full = {"configurable": {"session_id": "customer_001"}}

questions_full = [
    "안녕하세요. 어제 주문한 노트북 배송이 어디까지 왔는지 알고 싶어요.",
    "어제 말씀드린 노트북 모델명이 뭐였는지도 기억하시나요?",
]

print("=" * 60)
print("🧠 전체 버퍼 메모리 (모든 대화 이력 사용)")
print("=" * 60)

# 2단계: 두 번 질문하여 기억 여부 확인
for i, q in enumerate(questions_full, start=1):
    # full_buffer_chain: RunnableWithMessageHistory로 전체 히스토리 자동 관리
    answer = full_buffer_chain.invoke({"question": q}, config=config_full)
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer}\n")

🧠 전체 버퍼 메모리 (모든 대화 이력 사용)
1. [사용자] 안녕하세요. 어제 주문한 노트북 배송이 어디까지 왔는지 알고 싶어요.
   [AI] 안녕하세요! 어제 주문하신 노트북의 배송 상태를 확인해드리겠습니다. 주문 번호를 알려주시면, 배송 진행 상황을 확인해드릴 수 있습니다. 감사합니다!

2. [사용자] 어제 말씀드린 노트북 모델명이 뭐였는지도 기억하시나요?
   [AI] 죄송하지만, 이전 대화 내용을 기억할 수는 없어요. 노트북 모델명을 알려주시면, 배송 상태를 확인하는 데 도움이 될 것 같습니다. 모델명과 함께 주문 번호도 알려주시면 더욱 정확하게 확인해드릴 수 있습니다!



## 2. 윈도우 메모리 패턴 구현

이제 `ConversationBufferWindowMemory(k=3)`(구식 윈도우 메모리 클래스)에 해당하는 **Modern 윈도우 메모리 패턴**을 구현해요.

아이디어는 간단해요.

- 실제 히스토리(`ChatMessageHistory`)에는 **전체 대화**를 쌓아두고,
- 모델을 호출할 때만 `last_k_messages()` 함수로 **최근 k개의 메시지만** 잘라서 프롬프트에 넣어요.

### 윈도우 메모리 실행 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문<br/>session_id + question] --> CHAIN[window_chain 실행]
    CHAIN --> HIST[ChatMessageHistory 전체 조회]
    HIST --> TRIM[last_k_messages<br/>최근 k개 추출]
    TRIM --> PROMPT[prompt.invoke<br/>k개 메시지 주입]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> PARSER[StrOutputParser]
    PARSER --> SAVE_MANUAL[history.add_user/ai_message<br/>수동 저장]
    SAVE_MANUAL --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class CHAIN,TRIM,PROMPT,LLM,PARSER process
    class HIST,SAVE_MANUAL storage
    class ANSWER output
```

> 윈도우 체인에서는 `RunnableWithMessageHistory`를 사용하지 않고, 히스토리를 수동으로 저장해요(`add_user_message`, `add_ai_message`). 히스토리에는 전체 대화가 쌓이지만, 프롬프트에는 최근 k개만 들어가요.


In [3]:
# ---------------------------------------------------
# 윈도우 메모리 핵심 함수: last_k_messages, build_window_chain
# ---------------------------------------------------

from typing import List
from langchain_core.messages import BaseMessage


def last_k_messages(messages: List[BaseMessage], k: int = 6) -> List[BaseMessage]:
    """대화 메시지 리스트에서 마지막 k개만 가져오기.

    - 한 턴(human + ai)을 2개 메시지로 보면, k=6이면 최근 3턴 정도에 해당합니다.
    # 🎯 강의 포인트: k=6이 아니라 "메시지 6개" = "대화 3턴"입니다. 헷갈리기 쉬운 단위입니다.
    """
    # messages[-k:]: 리스트 슬라이싱으로 마지막 k개만 잘라냄
    return messages[-k:]


def build_window_chain(k_messages: int = 6):
    """최근 k개 메시지만 사용하는 윈도우 메모리 체인을 생성하는 헬퍼 함수."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        # 1) 세션 히스토리 전체 가져오기
        history = get_session_history(session_id)

        # 2) 최근 k개의 메시지만 선택 (윈도우 트리밍)
        # 💡 팁: 전체 히스토리는 보존하고, 프롬프트에 넣는 부분만 트리밍합니다
        window = last_k_messages(history.messages, k=k_messages)

        # 3) 윈도우 히스토리 + 현재 질문으로 프롬프트 구성
        prompt_msg = prompt.invoke({
            "chat_history": window,
            "question": question,
        })

        # 4) 모델 호출 + 문자열 파싱
        result = (model | StrOutputParser()).invoke(prompt_msg)

        # 5) 대화 기록에 현재 턴을 수동으로 추가 (RunnableWithMessageHistory 없이 직접 관리)
        # ⚠️ 주의: 이 저장 단계를 빠뜨리면 다음 턴에서 현재 대화를 기억하지 못합니다
        history.add_user_message(question)
        history.add_ai_message(result)

        return result

    return _chain


# 최근 6개 메시지만 사용하는 윈도우 체인 (약 3턴 분량)
window_chain = build_window_chain(k_messages=6)

## 3. 전체 버퍼 vs 윈도우 버퍼 비교 (고객 상담 시나리오)

동일한 고객 상담 대화를 두 체인에 순서대로 보내며, **전체 버퍼 체인**과 **윈도우 체인**의 응답 차이를 확인해요.

- `full_buffer_chain`: 모든 대화 메시지를 프롬프트에 넣어요. 대화가 길어질수록 토큰 사용량이 늘어나요.
- `window_chain`: 최근 k개 메시지만 사용해요. 토큰은 절약되지만, k보다 오래된 내용은 잊혀져요.

> 실무 팁: 상담 내용이 짧고 주제가 단순하면 윈도우 패턴이 좋아요. 복잡한 문제 해결 과정이나 이전 내용을 자주 참조해야 하는 경우에는 전체 버퍼나 요약 메모리(13번)를 고려하세요.


In [4]:
# ============================================================
# TODO: 전체 버퍼 vs 윈도우 체인 비교 실험을 수행하세요
# 힌트: 동일한 시나리오 메시지를 full_buffer_chain과 window_chain에 각각 실행하세요
# 예상 결과: 5번째 질문에서 full_buffer는 첫 번째 내용을 기억하고, window는 잊어야 합니다
# ============================================================

# ---------------------------------------------------
# 전체 버퍼 vs 윈도우 체인 비교 (고객 상담 시나리오)
# ---------------------------------------------------

# 비교 시나리오: 5개 질문으로 오래된 내용 기억 여부 차이 확인
scenario_messages = [
    "안녕하세요. 노트북 A 모델을 주문했는데, 배송 예정일이 어떻게 되나요?",
    "그럼 배송 주소를 회사로 바꾸고 싶은데 가능할까요?",
    "주소 변경 때문에 배송이 얼마나 지연될 수 있나요?",
    "혹시 키보드 레이아웃을 한/영 겸용으로 바꿀 수 있나요?",
    "제가 주문한 모델명이 정확히 뭐였는지도 다시 알려주세요.",
]

# 1단계: 전체 버퍼 체인 실행 (모든 대화 이력 사용)
print("=" * 60)
print("📦 전체 버퍼 체인 결과")
print("=" * 60)
config_full2 = {"configurable": {"session_id": "full_customer"}}

for i, q in enumerate(scenario_messages, start=1):
    answer = full_buffer_chain.invoke({"question": q}, config=config_full2)
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:100]}...\n")

# 2단계: 윈도우 체인 실행 (최근 k개만 사용)
print("\n" + "=" * 60)
print("🪟 윈도우 체인 결과 (최근 몇 턴만 사용)")
print("=" * 60)

for i, q in enumerate(scenario_messages, start=1):
    answer = window_chain({"session_id": "window_customer", "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:100]}...\n")

📦 전체 버퍼 체인 결과
1. [사용자] 안녕하세요. 노트북 A 모델을 주문했는데, 배송 예정일이 어떻게 되나요?
   [AI] 안녕하세요! 노트북 A 모델을 주문하셨군요. 배송 예정일은 주문하신 시점과 배송지에 따라 다를 수 있습니다. 주문 확인 이메일에 배송 추적 정보가 포함되어 있을 텐데요, 그 정보를...

2. [사용자] 그럼 배송 주소를 회사로 바꾸고 싶은데 가능할까요?
   [AI] 배송 주소 변경은 주문 상태에 따라 다를 수 있습니다. 만약 주문이 아직 처리되지 않았다면, 주소 변경이 가능할 수 있습니다. 고객 서비스 센터에 연락하시거나, 주문 확인 이메일에...

3. [사용자] 주소 변경 때문에 배송이 얼마나 지연될 수 있나요?
   [AI] 주소 변경으로 인한 배송 지연은 여러 요인에 따라 다를 수 있습니다. 일반적으로 주소 변경 요청이 처리된 후, 새로운 배송지로의 배송 준비가 필요하기 때문에 1~3일 정도 지연될 ...

4. [사용자] 혹시 키보드 레이아웃을 한/영 겸용으로 바꿀 수 있나요?
   [AI] 네, 대부분의 노트북에서는 키보드 레이아웃을 한/영 겸용으로 변경할 수 있습니다. 운영 체제에 따라 설정 방법이 다를 수 있는데요, 일반적으로 Windows에서는 다음과 같은 방법...

5. [사용자] 제가 주문한 모델명이 정확히 뭐였는지도 다시 알려주세요.
   [AI] 죄송하지만, 제가 고객님의 주문 내역을 직접 확인할 수는 없습니다. 주문 확인 이메일이나 고객 계정에서 모델명을 확인하실 수 있습니다. 만약 이메일을 찾기 어려우시거나 추가적인 도...


🪟 윈도우 체인 결과 (최근 몇 턴만 사용)
1. [사용자] 안녕하세요. 노트북 A 모델을 주문했는데, 배송 예정일이 어떻게 되나요?
   [AI] 안녕하세요! 노트북 A 모델을 주문하셨군요. 배송 예정일은 주문하신 시점과 배송 지역에 따라 다를 수 있습니다. 주문 확인 이메일에 배송 추적 정보가 포함되어 있을 텐데요, 그 정...

2. [사용자] 그럼 배송 주소를 회사로 바

## 핵심 요약

### 윈도우 메모리 핵심 구조

| 구성 요소 | 역할 |
|-----------|------|
| `ChatMessageHistory` | 전체 대화 이력 저장 (무제한 누적) |
| `last_k_messages(messages, k)` | 최근 k개 메시지만 추출 |
| `build_window_chain(k_messages)` | 윈도우 크기를 설정한 체인 생성 함수 |
| `history.add_user/ai_message()` | 윈도우 체인에서 수동 저장 |

### 전체 버퍼 vs 윈도우 비교

| 항목 | 전체 버퍼 (`full_buffer_chain`) | 윈도우 (`window_chain`) |
|------|--------------------------------|------------------------|
| 프롬프트 크기 | 대화가 쌓일수록 계속 증가 | 항상 최근 k개로 제한 |
| 오래된 내용 기억 | 가능 | 불가능 (k 초과 시 잊음) |
| 토큰 비용 | 증가 | 제어 가능 |
| 저장 자동화 | `RunnableWithMessageHistory` 자동 처리 | 수동 저장 필요 |

### 주의사항

- 한 턴(사용자 + AI)은 메시지 2개로 구성돼요. `k=6`이면 최근 **3턴**에 해당해요.
- 윈도우 체인은 `RunnableWithMessageHistory`를 사용하지 않아 히스토리를 수동으로 저장해야 해요. 저장을 빠뜨리면 다음 턴에서 이전 대화를 기억하지 못해요.
- 히스토리 자체는 전체 대화를 계속 쌓아요. 디스크 저장 시 장기적으로 공간이 커질 수 있어요.

### 다음 단계 예고

**13번**: 글자 수나 토큰 수를 기준으로 히스토리를 트리밍하는 **토큰 기반 메모리** 패턴과, LLM이 오래된 대화를 요약해 압축하는 **요약 메모리** 패턴을 다뤄요.


---

## 연습 문제

### 연습 1: 윈도우 크기(k)에 따른 기억력 차이 실험

윈도우 메모리에서 **k 값(최근 메시지 수)**을 바꿔가며 동일한 대화 시나리오를 실행하고, 기억력 차이를 비교해 보세요.

**요구사항:**
1. `build_window_chain(k_messages=2)`와 `build_window_chain(k_messages=10)`을 각각 만드세요 (k=2는 최근 1턴, k=10은 최근 5턴)
2. 아래 대화 시나리오를 두 체인에 동일하게 실행하세요:
   - "제 이름은 김민수이고, 서울에 살고 있어요."
   - "저는 백엔드 개발자로 3년째 일하고 있어요."
   - "요즘 Go 언어를 배우고 있어요."
   - "제 이름과 직업, 요즘 배우는 언어가 뭐였는지 정리해 주세요."
3. 마지막 질문에 대한 두 체인의 응답을 비교하고, k 값이 기억력에 어떤 영향을 미치는지 분석하세요

**힌트:**
- `build_window_chain`은 이미 노트북에 정의되어 있으므로, k 값만 바꿔서 새 체인을 만드세요
- 각 체인에 서로 다른 `session_id`를 사용해야 합니다 (예: `"small_k_user"`, `"large_k_user"`)

## 실전 예제: 윈도우 메모리 기반 챗봇

윈도우 메모리 패턴을 활용한 **완전한 챗봇**을 만들어 봅시다. 이 챗봇은 최근 k개의 메시지만 기억하며, 오래된 대화는 자연스럽게 잊어버립니다.

아래 예제에서는:
1. 5번의 대화를 나눈 후
2. 윈도우 밖으로 밀려난 첫 번째 대화 내용을 물어봐서
3. 실제로 "잊어버렸는지" 확인합니다

In [5]:
# ---------------------------------------------------
# 실전 예제: 윈도우 메모리 기반 여행 상담 챗봇
# ---------------------------------------------------
# 윈도우 크기 k=4 (최근 2턴만 기억하는 챗봇)

# 새로운 세션 스토어 (기존과 분리)
practical_store: dict[str, ChatMessageHistory] = {}


def get_practical_history(session_id: str) -> ChatMessageHistory:
    """실전 예제용 세션 히스토리 관리 함수."""
    if session_id not in practical_store:
        practical_store[session_id] = ChatMessageHistory()
    return practical_store[session_id]


# 여행 상담 전용 프롬프트
travel_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 친절한 여행 상담 챗봇입니다. "
        "이전 대화 맥락을 활용해 여행 계획을 도와주세요. "
        "사용자가 언급한 여행지, 일정, 예산 등을 기억하고 맞춤 추천을 해주세요.",
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])


def travel_chatbot(question: str, session_id: str = "travel_user", k: int = 4) -> str:
    """윈도우 메모리 기반 여행 상담 챗봇.

    Args:
        question: 사용자 질문
        session_id: 세션 식별자
        k: 윈도우 크기 (최근 k개 메시지만 사용)
    Returns:
        AI 응답 문자열
    """
    history = get_practical_history(session_id)

    # 최근 k개 메시지만 추출 (윈도우 트리밍)
    window = last_k_messages(history.messages, k=k)

    # 프롬프트 구성 및 모델 호출
    prompt_msg = travel_prompt.invoke({
        "chat_history": window,
        "question": question,
    })
    result = (model | StrOutputParser()).invoke(prompt_msg)

    # 전체 히스토리에 저장 (원본 보존)
    history.add_user_message(question)
    history.add_ai_message(result)

    return result


print("여행 상담 챗봇이 준비되었습니다! (윈도우 크기: k=4, 최근 2턴)")
print("=" * 60)

여행 상담 챗봇이 준비되었습니다! (윈도우 크기: k=4, 최근 2턴)


In [6]:
# ---------------------------------------------------
# 5번의 대화 후 윈도우 효과 확인
# ---------------------------------------------------

# 총 5번 대화 (k=4이므로 최근 2턴 = 4개 메시지만 기억)
conversations = [
    "안녕하세요! 다음 달에 제주도 여행을 계획하고 있어요. 예산은 100만원입니다.",
    "숙소는 해변 근처 펜션으로 알아보고 싶어요.",
    "렌터카도 빌릴 건데, 소형차로 충분할까요?",
    "맛집 추천도 해주세요. 해산물을 좋아해요.",
    "일정은 3박 4일로 잡았어요. 추천 코스를 짜주세요.",
]

print("=" * 60)
print("여행 상담 챗봇 대화 (총 5턴, 윈도우 k=4)")
print("=" * 60)

for i, q in enumerate(conversations, start=1):
    answer = travel_chatbot(q)
    print(f"\n--- 턴 {i} ---")
    print(f"[사용자] {q}")
    print(f"[챗봇] {answer[:200]}...")

# 현재 히스토리 상태 확인
history = get_practical_history("travel_user")
print(f"\n\n전체 저장된 메시지 수: {len(history.messages)}개")
print(f"윈도우에 포함되는 메시지 수: {min(4, len(history.messages))}개 (최근 {min(2, len(history.messages)//2)}턴)")

여행 상담 챗봇 대화 (총 5턴, 윈도우 k=4)

--- 턴 1 ---
[사용자] 안녕하세요! 다음 달에 제주도 여행을 계획하고 있어요. 예산은 100만원입니다.
[챗봇] 안녕하세요! 제주도 여행을 계획하고 계시다니 정말 멋진 선택이에요! 100만원의 예산으로 제주도에서 멋진 여행을 즐길 수 있을 거예요. 

여행 일정은 어떻게 되나요? 몇 일 정도 머무실 계획이신가요? 그리고 어떤 활동이나 장소를 방문하고 싶으신지도 알려주시면, 더 맞춤형으로 추천해드릴 수 있을 것 같아요!...

--- 턴 2 ---
[사용자] 숙소는 해변 근처 펜션으로 알아보고 싶어요.
[챗봇] 해변 근처의 펜션은 제주도에서 정말 좋은 선택이에요! 바다를 가까이에서 즐길 수 있으니까요. 예산 100만원으로 숙소와 식사, 교통비를 고려할 때, 다음과 같은 계획을 추천드릴게요.

### 숙소
- **해변 근처 펜션**: 제주도에는 다양한 해변 근처의 펜션이 있습니다. 예를 들어, 협재 해변이나 함덕 해변 근처의 펜션을 고려해보세요. 1박에 7~15만원...

--- 턴 3 ---
[사용자] 렌터카도 빌릴 건데, 소형차로 충분할까요?
[챗봇] 소형차는 제주도 여행에 아주 적합한 선택이에요! 제주도는 도로가 잘 정비되어 있고, 주요 관광지들이 소형차로 쉽게 접근할 수 있는 곳에 위치해 있기 때문에 소형차로도 충분히 편리하게 여행할 수 있습니다.

### 소형차의 장점:
1. **연비**: 소형차는 연비가 좋기 때문에 연료비를 절약할 수 있어요.
2. **주차**: 제주도는 관광지 주변에 주차 공간...

--- 턴 4 ---
[사용자] 맛집 추천도 해주세요. 해산물을 좋아해요.
[챗봇] 해산물을 좋아하신다면 제주도에서 정말 맛있는 해산물 맛집들이 많아요! 몇 군데 추천해드릴게요.

### 1. **우도 땅콩 아이스크림**
- **위치**: 우도
- **추천 메뉴**: 땅콩 아이스크림과 함께 신선한 해산물 요리
- **특징**: 우도의 신선한 해산물과 땅콩 아이스크림이 유명한 곳으로, 바다를 바

In [7]:
# ---------------------------------------------------
# 윈도우 효과 실증: 첫 번째 대화 내용을 물어보기
# ---------------------------------------------------
# k=4이므로 최근 2턴(4개 메시지)만 기억합니다.
# 첫 번째 턴의 "제주도", "예산 100만원" 정보는 이미 윈도우 밖으로 밀려났습니다.

print("=" * 60)
print("윈도우 효과 실증: 잊어버린 정보 확인")
print("=" * 60)

# 첫 번째 대화에서 언급한 정보를 물어봄 (윈도우 밖 → 기억 못함)
forgotten_q = "제가 처음에 말한 여행 예산이 얼마였는지 기억하시나요?"
answer_forgotten = travel_chatbot(forgotten_q)
print(f"\n[사용자] {forgotten_q}")
print(f"[챗봇] {answer_forgotten}")

# 최근 대화에서 언급한 정보를 물어봄 (윈도우 안 → 기억함)
remembered_q = "제가 며칠 일정이라고 했었죠?"
answer_remembered = travel_chatbot(remembered_q)
print(f"\n[사용자] {remembered_q}")
print(f"[챗봇] {answer_remembered}")

print("\n" + "=" * 60)
print("결론:")
print("- '예산 100만원'은 5턴 전에 말한 정보 → 윈도우(k=4) 밖 → 기억 못함")
print("- '3박 4일 일정'은 최근 대화 → 윈도우 안 → 기억함")
print("- 이것이 윈도우 메모리의 핵심 특성입니다!")
print("=" * 60)

윈도우 효과 실증: 잊어버린 정보 확인

[사용자] 제가 처음에 말한 여행 예산이 얼마였는지 기억하시나요?
[챗봇] 죄송하지만, 이전 대화에서 언급하신 여행 예산에 대한 정보는 기억하고 있지 않습니다. 예산을 알려주시면 그에 맞춰 여행 계획이나 추천을 조정해드릴 수 있습니다! 예산을 알려주시면 더욱 맞춤형으로 도와드릴게요.

[사용자] 제가 며칠 일정이라고 했었죠?
[챗봇] 사용자님께서 3박 4일 일정으로 여행을 계획하신다고 말씀하셨습니다. 추가로 궁금한 점이나 도움이 필요하신 부분이 있다면 언제든지 말씀해 주세요!

결론:
- '예산 100만원'은 5턴 전에 말한 정보 → 윈도우(k=4) 밖 → 기억 못함
- '3박 4일 일정'은 최근 대화 → 윈도우 안 → 기억함
- 이것이 윈도우 메모리의 핵심 특성입니다!


In [8]:
# ============================================================
# TODO: 윈도우 크기(k)에 따른 기억력 차이를 실험해보세요
# 힌트: build_window_chain(k_messages=2)와 build_window_chain(k_messages=10)을 만들어
#       동일한 시나리오를 실행하고 마지막 질문 응답을 비교하세요
# 예상 결과: k=2는 초반 정보를 잊고, k=10은 모든 정보를 기억해야 합니다
# ============================================================

# ---------------------------------------------------
# 연습 1 풀이: 윈도우 크기(k)에 따른 기억력 차이 실험
# ---------------------------------------------------
# ⚠️ 이전 셀의 store와 분리하기 위해 독립된 스토어를 사용합니다.
exercise_store: dict[str, ChatMessageHistory] = {}


def get_exercise_history(session_id: str) -> ChatMessageHistory:
    """연습 문제 전용 세션 히스토리 관리 함수."""
    if session_id not in exercise_store:
        exercise_store[session_id] = ChatMessageHistory()
    return exercise_store[session_id]


def build_exercise_chain(k_messages: int = 6):
    """연습 문제용 윈도우 메모리 체인 (독립 스토어 사용)."""

    def _chain(inputs: dict) -> str:
        session_id: str = inputs["session_id"]
        question: str = inputs["question"]

        history = get_exercise_history(session_id)
        window = last_k_messages(history.messages, k=k_messages)

        # 프롬프트 → 모델 → 파서를 하나의 체인으로 구성
        chain = prompt | model | StrOutputParser()
        result = chain.invoke({
            "chat_history": window,
            "question": question,
        })

        history.add_user_message(question)
        history.add_ai_message(result)
        return result

    return _chain


# 1단계: 작은 윈도우 체인 (k=2: 최근 메시지 2개 = 약 1턴)
small_window_chain = build_exercise_chain(k_messages=2)

# 2단계: 큰 윈도우 체인 (k=10: 최근 메시지 10개 = 약 5턴)
large_window_chain = build_exercise_chain(k_messages=10)

# 테스트 시나리오: 4개 대화 중 마지막에 앞서 말한 내용을 물어봄
scenario = [
    "제 이름은 김민수이고, 서울에 살고 있어요.",
    "저는 백엔드 개발자로 3년째 일하고 있어요.",
    "요즘 Go 언어를 배우고 있어요.",
    "제 이름과 직업, 요즘 배우는 언어가 뭐였는지 정리해 주세요.",
]

# 3단계: 작은 윈도우 (k=2) 실행 → 오래된 정보 기억 못함
print("=" * 60)
print("[ 작은 윈도우 (k=2, 최근 1턴) ]")
print("=" * 60)
for i, q in enumerate(scenario, start=1):
    answer = small_window_chain({"session_id": "small_k_user", "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:150]}...\n")

# 4단계: 큰 윈도우 (k=10) 실행 → 모든 정보 기억
print("=" * 60)
print("[ 큰 윈도우 (k=10, 최근 5턴) ]")
print("=" * 60)
for i, q in enumerate(scenario, start=1):
    answer = large_window_chain({"session_id": "large_k_user", "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:150]}...\n")

# 5단계: 비교 분석 출력
print("=" * 60)
print("[ 비교 분석 ]")
print("=" * 60)
print("k=2 (최근 1턴):")
print("  - 바로 직전 턴의 내용만 기억합니다.")
print("  - 마지막 질문에서 이름/직업 등 초반 정보를 기억하지 못할 가능성이 높습니다.")
print()
print("k=10 (최근 5턴):")
print("  - 4턴 분량의 대화를 모두 기억합니다.")
print("  - 마지막 질문에서 이름, 직업, 학습 중인 언어를 모두 정리할 수 있습니다.")
print()
print("결론: k 값이 작으면 토큰을 절약하지만 맥락이 부족해지고,")
print("      k 값이 크면 맥락이 풍부하지만 토큰 사용량이 증가합니다.")

[ 작은 윈도우 (k=2, 최근 1턴) ]
1. [사용자] 제 이름은 김민수이고, 서울에 살고 있어요.
   [AI] 안녕하세요, 김민수님! 서울에 사신다니 멋지네요. 서울에서 어떤 활동을 즐기시나요?...

2. [사용자] 저는 백엔드 개발자로 3년째 일하고 있어요.
   [AI] 백엔드 개발자로 3년째 일하고 계시다니 대단하네요! 어떤 기술 스택을 주로 사용하시나요? 또는 최근에 흥미롭게 진행한 프로젝트가 있으신가요?...

3. [사용자] 요즘 Go 언어를 배우고 있어요.
   [AI] Go 언어를 배우고 계시다니 멋지네요! Go는 성능이 뛰어나고 동시성을 잘 처리할 수 있어서 백엔드 개발에 많이 사용되죠. 어떤 부분이 가장 흥미롭거나 도전적이었나요? 혹시 Go로 진행하고 싶은 프로젝트가 있으신가요?...

4. [사용자] 제 이름과 직업, 요즘 배우는 언어가 뭐였는지 정리해 주세요.
   [AI] 물론입니다! 

- **이름**: (이름을 입력해 주세요)
- **직업**: (직업을 입력해 주세요)
- **요즘 배우는 언어**: Go 언어

이렇게 정리할 수 있습니다. 추가로 더 알고 싶은 정보가 있으면 말씀해 주세요!...

[ 큰 윈도우 (k=10, 최근 5턴) ]
1. [사용자] 제 이름은 김민수이고, 서울에 살고 있어요.
   [AI] 안녕하세요, 김민수님! 서울에 사신다니 멋지네요. 서울에서 어떤 활동을 즐기시나요?...

2. [사용자] 저는 백엔드 개발자로 3년째 일하고 있어요.
   [AI] 백엔드 개발자로 3년째 일하고 계시다니 대단하네요! 어떤 기술 스택을 주로 사용하시나요? 또는 최근에 흥미롭게 작업한 프로젝트가 있나요?...

3. [사용자] 요즘 Go 언어를 배우고 있어요.
   [AI] Go 언어를 배우고 계시다니 좋네요! Go는 성능이 뛰어나고 간결한 문법 덕분에 많은 개발자들에게 인기가 많죠. 어떤 부분이 가장 흥미롭거나 도전적이었나요?...

4. [사용자] 제 이름과 직업, 요즘 배우는 언어가 뭐였는지 정리해 주세요.
  